# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [ ]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE***Chart choice & justification:** **Chart choice & justification:** I used a line chart because the question is about change over time. Setting every continent's 1970 population to 100 makes growth comparable across continents of very different sizes. Africa is highlighted in blue and the others are grey, so the key finding stands out.

**Finding:** Africa grew fastest. Its index hit about 390 by 2022, nearly quadrupling since 1970. Oceania (about 231) and South America (about 226) followed, but both only a bit more than doubled.

In [1]:
# Q1
import pandas as pd
import plotly.express as px

# Load the quiz data
population = pd.read_csv("world_population.csv")

years = [1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022]
year_columns = [f"{year} Population" for year in years]

continent_totals = (
    population.groupby("Continent", as_index=False)[year_columns]
    .sum()
)

indexed = continent_totals.melt(
    id_vars="Continent",
    var_name="Year",
    value_name="Population",
)
indexed["Year"] = indexed["Year"].str.extract(r"(\d{4})").astype(int)
indexed["Index"] = (
    indexed.groupby("Continent", sort=False)["Population"].transform(
        lambda values: values / values.iloc[0] * 100
    )
)

fig = px.line(
    indexed,
    x="Year",
    y="Index",
    color="Continent",
    category_orders={"Continent": ["Africa", "Asia", "Europe", "North America", "Oceania", "South America"]},
    template="plotly_white",
)

context_colour = "#A6A6A6"
accent_colour = "#0072B2"
for trace in fig.data:
    if trace.name == "Africa":
        trace.update(line={"color": accent_colour, "width": 4}, mode="lines+markers")
    else:
        trace.update(line={"color": context_colour, "width": 2}, mode="lines")

fig.update_layout(
    title="Africa leads: its population reached 390% of 1970 by 2022",
    yaxis_title="",
    xaxis_title="",
    legend={"title": None, "orientation": "h", "y": -0.22, "x": 0.5, "xanchor": "center"},
    hovermode="x unified",
    margin={"l": 55, "r": 35, "t": 75, "b": 100},
)
fig.update_xaxes(showgrid=False, tickmode="linear", dtick=10)
fig.update_yaxes(showgrid=False, ticksuffix="", range=[90, 400])

africa_2022 = indexed[(indexed["Continent"] == "Africa") & (indexed["Year"] == 2022)]["Index"].iloc[0]
fig.add_annotation(
    x=2022,
    y=africa_2022,
    text="Africa: 390",
    showarrow=True,
    arrowhead=2,
    ax=45,
    ay=-25,
    font={"color": accent_colour, "size": 12},
)

fig.show()


---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE***Chart choice & justification:** A horizontal bar chart is the best way to compare population growth in 15 countries. The countries are arranged from highest to lowest population increase, making them easy to compare. Blue represents the four African countries, while grey represents the other countries. Each bar shows the population increase in millions, so there is no need for a separate legend.

**Finding:** India and China had the biggest population increases between 1970 and 2022, adding around 860 million and 603 million people. Out of the 15 countries, **4 are in Africa: Nigeria, Ethiopia, DR Congo, and Egypt.**

In [2]:
# Q2
import pandas as pd
import plotly.express as px

population = pd.read_csv("world_population.csv")
population["Population Added"] = population["2022 Population"] - population["1970 Population"]
top15 = population.nlargest(15, "Population Added").copy()
top15["Africa"] = top15["Continent"].eq("Africa")
top15["Added (millions)"] = top15["Population Added"] / 1_000_000

top15 = top15.sort_values("Population Added", ascending=True)

fig = px.bar(
    top15,
    x="Added (millions)",
    y="Country/Territory",
    color="Africa",
    orientation="h",
    text="Added (millions)",
    template="plotly_white",
    color_discrete_map={True: "#0072B2", False: "#A6A6A6"},
)

fig.update_traces(texttemplate="%{text:.0f}M", textposition="outside", cliponaxis=False)
fig.update_layout(
    title="India and China dominate: 4 of the top 15 gains are in Africa",
    xaxis_title="",
    yaxis_title="",
    showlegend=False,
    bargap=0.28,
    margin={"l": 115, "r": 55, "t": 75, "b": 55},
)
fig.update_xaxes(showgrid=False, range=[0, 920], ticksuffix="M")
fig.update_yaxes(showgrid=False)

fig.add_annotation(
    x=0.99,
    y=0.02,
    xref="paper",
    yref="paper",
    text="Blue = Africa (4 countries)",
    showarrow=False,
    xanchor="right",
    font={"color": "#0072B2", "size": 12},
)

fig.show()


---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE***Chart choice & justification:** The scatter plot shows the relationship between a country's land area and population, using different colours for each continent. &#x20;

**Finding:** Countries with larger areas generally have larger populations (correlation 0.87). China and India have very large populations, while Russia, Canada, and Australia have large land areas but smaller populations.

In [3]:
# Q3
import pandas as pd
import plotly.express as px

population = pd.read_csv("world_population.csv")

continent_colors = {
    "Africa": "#0072B2",
    "Asia": "#E69F00",
    "Europe": "#CC79A7",
    "North America": "#56B4E9",
    "Oceania": "#7F7F7F",
    "South America": "#8C6D31",
}

fig = px.scatter(
    population,
    x="Area (km²)",
    y="2022 Population",
    color="Continent",
    hover_name="Country/Territory",
    hover_data={"Area (km²)": ":,.0f", "2022 Population": ":,.0f", "Continent": True},
    log_x=True,
    log_y=True,
    template="plotly_white",
    color_discrete_map=continent_colors,
)

fig.update_traces(marker={"size": 8, "line": {"width": 0.5, "color": "white"}})
fig.update_layout(
    title="Larger countries tend to have more people, but dense Asian states break the pattern",
    xaxis_title="",
    yaxis_title="",
    legend={"title": None, "orientation": "h", "y": -0.18, "x": 0.5, "xanchor": "center"},
    margin={"l": 75, "r": 35, "t": 75, "b": 105},
)
fig.update_xaxes(showgrid=False, ticksuffix="", title="")
fig.update_yaxes(showgrid=False, ticksuffix="", title="")

outliers = ["Russia", "Canada", "Australia", "China", "India", "Bangladesh", "Japan"]
for country in outliers:
    row = population.loc[population["Country/Territory"] == country].iloc[0]
    fig.add_annotation(
        x=row["Area (km²)"],
        y=row["2022 Population"],
        text=country,
        showarrow=True,
        arrowhead=2,
        ax=35 if country in ["China", "India", "Bangladesh", "Japan"] else -35,
        ay=-25,
        font={"size": 11},
    )

fig.add_annotation(
    x=0.02,
    y=0.98,
    xref="paper",
    yref="paper",
    text="Overall log-scale correlation: 0.87",
    showarrow=False,
    xanchor="left",
    yanchor="top",
    font={"size": 12, "color": "#505050"},
)

fig.show()


---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE***Chart choice & justification:** The box plot compares population density across continents, showing the median, spread, and extreme values. The log scale makes it easier to compare continents with very different population densities.

**Finding:** North America has the highest median density (221 people/km²), while South America has the lowest (18 people/km²). Macau and Monaco have the highest extreme values, with around 23,172 and 18,235 people/km², respectively.

In [5]:
# Q4
import pandas as pd
import plotly.express as px

population = pd.read_csv("world_population.csv")
continent_order = [
    "South America", "Africa", "Oceania", "Asia", "Europe", "North America"
]
continent_colors = {
    "Africa": "#A6A6A6",
    "Asia": "#A6A6A6",
    "Europe": "#A6A6A6",
    "North America": "#0072B2",
    "Oceania": "#A6A6A6",
    "South America": "#A6A6A6",
}

fig = px.box(
    population,
    x="Density (per km²)",
    y="Continent",
    category_orders={"Continent": continent_order},
    color="Continent",
    color_discrete_map=continent_colors,
    log_x=True,
    points="outliers",
    hover_name="Country/Territory",
    template="plotly_white",
)

fig.update_traces(
    marker={"size": 5, "opacity": 0.7},
    line={"width": 1.5},
    boxpoints="outliers",
)
fig.update_layout(
    title="North America has the highest median; Asian microstates reach the highest densities",
    xaxis_title="Population density (people/km²)",
    yaxis_title="",
    showlegend=False,
    margin={"l": 110, "r": 45, "t": 75, "b": 65},
)
fig.update_xaxes(showgrid=False, ticks="outside", tickformat="~s")
fig.update_yaxes(showgrid=False)

fig.show()


---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE**A choropleth map is the best choice because it shows which countries experienced population growth or decline between 2000 and 2022. Different colours represent population decreases, little change, and increases.

**Finding:** Out of 234 countries and territories, 200 experienced population growth, while 34 saw a decline. Most declines happened in Europe. Qatar had the highest growth (+317%), while the Northern Mariana Islands had the biggest decline (−38%).

In [6]:
# Q5
import pandas as pd
import plotly.express as px

population = pd.read_csv("world_population.csv")
population["Change (%)"] = (
    population["2022 Population"] / population["2000 Population"] - 1
) * 100

# Cap only the color scale so a few large increases do not flatten most countries.
# Exact percentage changes remain available on hover.
zero_position = 40 / (40 + 150)
change_colors = [
    [0, "#D55E00"],
    [zero_position, "#F0F0F0"],
    [1, "#0072B2"],
]

fig = px.choropleth(
    population,
    locations="CCA3",
    locationmode="ISO-3",
    color="Change (%)",
    hover_name="Country/Territory",
    hover_data={
        "Continent": True,
        "Change (%)": ":+.1f",
        "2000 Population": ":,.0f",
        "2022 Population": ":,.0f",
        "CCA3": False,
    },
    color_continuous_scale=change_colors,
    range_color=(-40, 150),
    template="plotly_white",
)

fig.update_traces(marker_line_color="white", marker_line_width=0.35)
fig.update_layout(
    title={
        "text": "200 of 234 countries and territories grew; declines cluster in Europe<br><sup>Colors saturate above +150%; hover for exact percentage change</sup>"
    },
    margin={"l": 10, "r": 10, "t": 90, "b": 10},
    coloraxis_colorbar={
        "title": "Change (%)",
        "tickvals": [-40, 0, 50, 100, 150],
        "ticksuffix": "%",
    },
)
fig.update_geos(
    projection_type="natural earth",
    showframe=False,
    showcoastlines=False,
    showcountries=True,
    countrycolor="white",
    showland=True,
    landcolor="#E5E5E5",
    showocean=True,
    oceancolor="#F7F7F7",
)

fig.show()


---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE***Chart choice & justification:** A treemap shows the world population by continent and country. The size of each rectangle represents its population, with Asia highlighted in blue and the other continents in grey.

**Finding:** The total world population in 2022 was about **7.97 billion**. Asia had the largest share (59.2%), followed by Africa (17.9%). China and India together made up **35.7%** of the world's population.

In [7]:
# Q6
import pandas as pd
import plotly.express as px

population = pd.read_csv("world_population.csv")
continent_order = [
    "Asia", "Africa", "Europe", "North America", "South America", "Oceania"
]
population["_continent_order"] = population["Continent"].map(
    {continent: order for order, continent in enumerate(continent_order)}
)
population = population.sort_values(
    ["_continent_order", "2022 Population"], ascending=[True, False]
).drop(columns="_continent_order")
continent_colors = {
    "Asia": "#0072B2",
    "Africa": "#A6A6A6",
    "Europe": "#A6A6A6",
    "North America": "#A6A6A6",
    "South America": "#A6A6A6",
    "Oceania": "#A6A6A6",
}

fig = px.treemap(
    population,
    path=[px.Constant("World"), "Continent", "Country/Territory"],
    values="2022 Population",
    color="Continent",
    color_discrete_map=continent_colors,
    template="plotly_white",
)

fig.update_traces(
    sort=False,
    texttemplate="%{label}<br>%{percentParent:.1%}",
    hovertemplate=(
        "<b>%{label}</b><br>Population: %{value:,.0f}"
        "<br>Share of parent: %{percentParent:.1%}"
        "<br>Share of world: %{percentRoot:.1%}<extra></extra>"
    ),
    marker_line_color="white",
    marker_line_width=1,
    root_color="#F0F0F0",
)
fig.update_layout(
    title="Asia holds 59% of the world's population; China and India together account for 36%",
    margin={"l": 15, "r": 15, "t": 75, "b": 15},
    uniformtext={"minsize": 9, "mode": "hide"},
)

fig.show()


---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE***Chart choice & justification:*A 100% stacked bar chart shows how the world's population is divided between four regions over time. Each bar adds up to 100%, making it easy to compare changes in population share.

**Finding:** Africa's share increased from 9.9% to 17.9%, while Europe's share decreased from 17.8% to 9.3% between 1970 and 2022. Asia remained almost stable, changing from 58.1% to 59.2%.

In [8]:
# Q7
import pandas as pd
import plotly.express as px

years = [1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022]
year_columns = [f"{year} Population" for year in years]
population = pd.read_csv("world_population.csv")
continent_totals = population.groupby("Continent", as_index=False)[year_columns].sum()

shares = continent_totals.melt(
    id_vars="Continent",
    var_name="Year",
    value_name="Population",
)
shares["Year"] = shares["Year"].str.extract(r"(\d{4})").astype(int)
shares["Group"] = shares["Continent"].where(
    shares["Continent"].isin(["Africa", "Asia", "Europe"]),
    "Rest of the world",
)
shares = shares.groupby(["Year", "Group"], as_index=False)["Population"].sum()
shares["World total"] = shares.groupby("Year")["Population"].transform("sum")
shares["Share (%)"] = shares["Population"] / shares["World total"] * 100
shares["Year"] = shares["Year"].astype(str)

fig = px.bar(
    shares,
    x="Year",
    y="Share (%)",
    color="Group",
    category_orders={
        "Year": [str(year) for year in years],
        "Group": ["Africa", "Asia", "Europe", "Rest of the world"],
    },
    color_discrete_map={
        "Africa": "#0072B2",
        "Asia": "#E69F00",
        "Europe": "#CC79A7",
        "Rest of the world": "#A6A6A6",
    },
    barmode="stack",
    template="plotly_white",
)

fig.update_traces(
    marker_line_color="white",
    marker_line_width=0.8,
    hovertemplate="%{x}<br>%{fullData.name}: %{y:.1f}%<extra></extra>",
)
fig.update_layout(
    title="Africa's share is rising while Europe's is falling; Asia remains near 60%",
    xaxis_title="",
    yaxis_title="World population share",
    legend={"title": None, "orientation": "h", "y": -0.2, "x": 0.5, "xanchor": "center"},
    margin={"l": 65, "r": 25, "t": 75, "b": 95},
    bargap=0.18,
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False, range=[0, 100], ticksuffix="%", dtick=20)

fig.show()
